# SLIIT IT3051 - Data Mining & Predictive Analytics
## EV Battery Prognostics & Health Management (PHM) Dual-Task System
### Group: Necrons | Phase 1: EDA, Data Cleaning & Preprocessing (Week 1 / Viva 1)

---

### Team Roles & Modular Work Breakdown:
| Person | Topic Owned | Key Deliverables |
| :--- | :--- | :--- |
| **Person A** | **Data Structure, Schema & Missingness Audit** | Schema inspection, variable types, duplicate checks, 67-column missingness breakdown, sensor sanity screening |
| **Person B** | **Distributions, Outliers & Treatment Decisions** | Univariate distributions, RUL bell-curve analysis, IQR/Z-score outlier detection, physical outlier justification |
| **Person C** | **Imbalance, Multicollinearity & Feature Engineering** | Class imbalance (18,616 vs 1,384), correlation heatmaps, bivariate degradation analysis, 4 domain features |
| **Person D** | **Data Leakage Guard, Splits & ColumnTransformer** | Target Isolation, ID exclusion, stratified train/test split, production ColumnTransformer pipeline |


### 0. Environment Setup & Global Imports


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'Arial'

print('Libraries loaded successfully!')


#### C.1 Task 2 Class Imbalance Analysis


In [ ]:
# Task 2 Class Distribution
fail_counts = df_raw['battery_failure'].value_counts()
print(fail_counts)

plt.figure(figsize=(7, 5))
bars = plt.bar(['Normal (0)', 'Failure (1)'], [fail_counts[0], fail_counts[1]], color=['#2a9d8f', '#e63946'], width=0.45)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 300, f'{yval:,}\n({yval/len(df_raw)*100:.2f}%)', ha='center', va='bottom', fontweight='bold')

plt.title('Task 2 Target: Critical Battery Failure Class Imbalance', fontsize=12, fontweight='bold')
plt.ylabel('Record Count')
plt.ylim(0, 21500)
plt.show()

print(f'Imbalance Ratio: {fail_counts[0]/fail_counts[1]:.2f} : 1')


#### C.2 Physical Variable Correlation & Degradation Patterns


In [ ]:
# Correlation of physical metrics with targets
key_features = [
    'internal_resistance', 'cell_temperature_max', 'cell_temperature_avg',
    'voltage_imbalance', 'capacity_loss_percent', 'state_of_health',
    'predicted_remaining_life_cycles', 'battery_failure'
]
corr_sub = df_raw[key_features].dropna().corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_sub, annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar_kws={'label': 'Pearson Correlation'})
plt.title('Physical Feature Correlation Matrix', fontsize=12, fontweight='bold')
plt.show()


#### C.3 Bivariate Physical Discrimination (Normal vs Failure)


In [ ]:
# Boxplots of key failure indicators
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(x='battery_failure', y='internal_resistance', data=df_raw, ax=axes[0], palette=['#457b9d', '#e63946'], hue='battery_failure', legend=False)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Normal (0)', 'Failure (1)'])
axes[0].set_title('Internal Resistance by Failure Status', fontweight='bold')

sns.boxplot(x='battery_failure', y='cell_temperature_max', data=df_raw, ax=axes[1], palette=['#457b9d', '#e63946'], hue='battery_failure', legend=False)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Normal (0)', 'Failure (1)'])
axes[1].set_title('Max Cell Temperature by Failure Status', fontweight='bold')

sns.boxplot(x='battery_failure', y='voltage_imbalance', data=df_raw, ax=axes[2], palette=['#457b9d', '#e63946'], hue='battery_failure', legend=False)
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(['Normal (0)', 'Failure (1)'])
axes[2].set_title('Voltage Imbalance by Failure Status', fontweight='bold')

plt.tight_layout()
plt.show()


#### C.4 Stage 3 Feature Engineering: Domain-Specific Electrochemical Features


In [ ]:
df_feat = df_raw.copy()

# 1. Thermal Gradient
df_feat['temp_delta'] = df_feat['cell_temperature_max'] - df_feat['cell_temperature_avg']

# 2. Electrical Stress Ratio (C-Rate Proxy)
df_feat['c_rate_proxy'] = df_feat['average_charge_power_kw'] / (df_feat['battery_capacity_kwh'] + 1e-5)

# 3. Cell Degradation Index (Relative Voltage Spread)
df_feat['cell_voltage_spread'] = df_feat['cell_voltage_std'] / (df_feat['cell_voltage_avg'] + 1e-5)

# 4. Aggressive Driving Usage Stress Metric
df_feat['stress_index'] = df_feat['aggressive_acceleration_score'] * df_feat['hard_braking_score']

eng_cols = ['temp_delta', 'c_rate_proxy', 'cell_voltage_spread', 'stress_index']
print('=== ENGINEERED DOMAIN FEATURES STATISTICAL SUMMARY ===')
print(df_feat[eng_cols].describe().round(4).T[['mean', 'std', 'min', '50%', 'max']])

# Correlation with targets
eng_corrs = df_feat[eng_cols + ['predicted_remaining_life_cycles', 'battery_failure']].corr()
print('\nCorrelations with Dual Targets:')
print(pd.DataFrame({
    'Task 1 (RUL Corr)': eng_corrs['predicted_remaining_life_cycles'][eng_cols],
    'Task 2 (Failure Corr)': eng_corrs['battery_failure'][eng_cols]
}))


**Person C Viva Notes:**
- Task 2 has a severe class imbalance of 93.08% normal to 6.92% failure (13.45:1).
- Accuracy is an inappropriate evaluation metric (predicting all 0s gives 93.08% accuracy while missing all catastrophic failures). We must prioritize **Recall** and **PR-AUC**.
- The 4 engineered features capture thermal gradients, charging stress (C-rate), voltage non-uniformity, and mechanical drive cycle shock.


---
# Section D: Data Leakage Guard, Train/Test Split & ColumnTransformer Pipeline
**Owner: Person D**
**Focus:** Boundary conditions, Target Isolation, Stratified train/test splitting, production ColumnTransformer preprocessing, and data leakage verification.
